# Detecção de árvores (DeepForest) + cobertura verde — Praia de Icaraí, Niterói/RJ

**Importante:** o DeepForest detecta **copas de árvores individuais** (caixas), não faz
"classificação de áreas verdes" no sentido de segmentar vegetação contínua. Ele foi treinado
em RGB de altíssima resolução (~0,1 m, tipo NAIP). Por isso este notebook faz duas coisas:

1. **DeepForest** → detecta árvores individuais na cena (o que ele de fato faz).
2. **Índice de vegetação RGB (ExG + Otsu)** → estima a **fração de cobertura verde**
   (mato, gramado, mata), que é o que responde melhor a "área verde".

A imagem vem do **Esri World Imagery** (alta resolução) via `contextily`, georreferenciada em
GeoTIFF (EPSG:3857) para que as saídas saiam com coordenadas. Sentinel-2/CBERS a 10 m **não**
servem para o DeepForest.

> Se você tiver um ortomosaico próprio de Niterói (~0,1–0,3 m), pule o download e aponte
> `TIF_PATH` para o seu arquivo — o resultado será muito melhor.

## 1. Instalação

In [ ]:
!pip install -q deepforest contextily rasterio geopandas scikit-image

## 2. Área de interesse — Praia de Icaraí

Recorte fechado no arco da praia + calçadão + primeira faixa da Av. Jornalista Alberto Francisco
Torres. **A areia não tem árvore** — as detecções se concentram nas árvores do calçadão e da
avenida. Ajuste `WEST/SOUTH/EAST/NORTH` (lon/lat) e `ZOOM`. `ZOOM=19` ≈ 0,27 m/px (melhor pro
DeepForest); `ZOOM=18` ≈ 0,55 m/px (mais leve).

In [ ]:
# Bounding box em lon/lat (WGS84) — arco da Praia de Icarai, Niteroi/RJ
# (pin da praia: -22.9079, -43.1124 na ponta oeste; arco segue a leste ate Sao Francisco)
WEST, SOUTH, EAST, NORTH = -43.113, -22.911, -43.099, -22.905
ZOOM = 19          # 18 = mais leve, 19 = mais resolucao
TIF_PATH = "icarai_praia_rgb.tif"

# Estimativa grosseira de tamanho
import math
res = 156543.03 * math.cos(math.radians((SOUTH + NORTH) / 2)) / (2 ** ZOOM)
print(f"Resolucao aprox.: {res:.2f} m/px  (zoom {ZOOM})")

## 3. Baixar imagem RGB e salvar GeoTIFF georreferenciado

In [ ]:
import contextily as cx
import numpy as np
import rasterio
from rasterio.transform import from_bounds

# Baixa mosaico do Esri World Imagery; extent volta em Web Mercator (EPSG:3857)
img, extent = cx.bounds2img(
    WEST, SOUTH, EAST, NORTH,
    zoom=ZOOM,
    source=cx.providers.Esri.WorldImagery,
    ll=True,               # bounds de entrada em lon/lat
)
left, right, bottom, top = extent          # em 3857
rgb = img[:, :, :3].astype("uint8")        # descarta alpha
h, w = rgb.shape[:2]
print("Imagem:", rgb.shape)

transform = from_bounds(left, bottom, right, top, w, h)
with rasterio.open(
    TIF_PATH, "w", driver="GTiff",
    height=h, width=w, count=3, dtype="uint8",
    crs="EPSG:3857", transform=transform,
) as dst:
    for i in range(3):
        dst.write(rgb[:, :, i], i + 1)

print("Salvo:", TIF_PATH)

## 4. Carregar o modelo DeepForest (com fallback de versão)

In [ ]:
from deepforest import main

model = main.deepforest()
try:
    # DeepForest >= 2.0
    model.load_model("weecology/deepforest-tree")
    print("Modelo carregado via load_model (>=2.0)")
except Exception as e:
    # DeepForest < 2.0
    model.use_release()
    print("Modelo carregado via use_release (<2.0)")

## 5. Detecção de árvores (predict_tile)

Divide a cena em janelas de 400 px com sobreposição. Ajuste `patch_size`/`patch_overlap` e o
corte de confiança (`score`) conforme o resultado.

In [ ]:
try:
    # API nova: path=
    pred = model.predict_tile(path=TIF_PATH, patch_size=400, patch_overlap=0.25)
except TypeError:
    # API antiga: raster_path=
    pred = model.predict_tile(raster_path=TIF_PATH, patch_size=400, patch_overlap=0.25)

# Filtro opcional por confianca
pred = pred[pred["score"] >= 0.2].reset_index(drop=True)
print(f"Arvores detectadas: {len(pred)}")
pred.head()

## 6. Georreferenciar as caixas e exportar GeoJSON

As caixas do DeepForest vêm em pixels. Aqui convertemos para coordenadas usando o transform do
GeoTIFF, e reprojetamos para WGS84 (EPSG:4326).

In [ ]:
import geopandas as gpd
from shapely.geometry import box as shbox

with rasterio.open(TIF_PATH) as src:
    T = src.transform
    src_crs = src.crs

def to_geom(r):
    x_ul, y_ul = rasterio.transform.xy(T, r.ymin, r.xmin, offset="ul")
    x_lr, y_lr = rasterio.transform.xy(T, r.ymax, r.xmax, offset="lr")
    return shbox(min(x_ul, x_lr), min(y_ul, y_lr), max(x_ul, x_lr), max(y_ul, y_lr))

gdf = gpd.GeoDataFrame(
    pred.copy(),
    geometry=[to_geom(r) for r in pred.itertuples()],
    crs=src_crs,
).to_crs(4326)

gdf.to_file("arvores_icarai_praia.geojson", driver="GeoJSON")
print("Salvo: arvores_icarai_praia.geojson  |  centroides em WGS84:")
gdf.head()[["score", "geometry"]]

## 7. Visualizar as detecções sobre a imagem

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(rgb)
for r in pred.itertuples():
    ax.add_patch(patches.Rectangle(
        (r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin,
        fill=False, edgecolor="yellow", linewidth=0.8,
    ))
ax.set_title(f"DeepForest — {len(pred)} arvores detectadas")
ax.axis("off")
plt.show()

## 8. Cobertura verde (índice de vegetação RGB)

Como o DeepForest não segmenta "área verde", aqui um proxy simples e reprodutível: **Excess
Green (ExG = 2G − R − B)** + limiar de **Otsu**. Retorna a **fração de pixels verdes** e salva a
máscara como GeoTIFF. Para vegetação, um NDVI (com banda NIR) seria mais robusto — mas exige
imagem multiespectral, que o RGB do basemap não tem.

In [ ]:
from skimage.filters import threshold_otsu

R = rgb[:, :, 0].astype(float)
G = rgb[:, :, 1].astype(float)
B = rgb[:, :, 2].astype(float)

exg = 2 * G - R - B
thr = threshold_otsu(exg)
green = exg > thr
frac = float(green.mean())
print(f"Cobertura verde estimada: {frac*100:.1f}% da area")

# Salva mascara georreferenciada
with rasterio.open(
    "verde_icarai_praia.tif", "w", driver="GTiff",
    height=green.shape[0], width=green.shape[1], count=1, dtype="uint8",
    crs="EPSG:3857", transform=transform,
) as dst:
    dst.write(green.astype("uint8") * 255, 1)

fig, axs = plt.subplots(1, 2, figsize=(16, 8))
axs[0].imshow(rgb);              axs[0].set_title("RGB");          axs[0].axis("off")
axs[1].imshow(green, cmap="Greens"); axs[1].set_title(f"Mascara verde ({frac*100:.1f}%)"); axs[1].axis("off")
plt.show()

## 9. Baixar resultados / próximos passos

```python
from google.colab import files
files.download("arvores_icarai_praia.geojson")
files.download("verde_icarai_praia.tif")
```

**Para melhorar:**
- Use **ortofoto própria** de ~0,1–0,3 m em vez do basemap → aponte `TIF_PATH` e pule os passos 2–3.
- Combine as copas com um **CHM (LiDAR)** para filtrar falsos positivos e estimar altura — como
  na sua abordagem de fusão DeepForest + CHM em Icaraí. No calçadão isso ajuda a separar árvore
  de guarda-sol/quiosque, que o RGB sozinho pode confundir.
- Ajuste `patch_size`, `patch_overlap` e o corte de `score` para a resolução real da sua cena.
- Para "área verde" de verdade, considere segmentação semântica (ex.: um modelo U-Net) ou NDVI
  com imagem multiespectral (Sentinel-2/PlanetScope), tratando o DeepForest só como a camada
  de árvores individuais.